In [ ]:
import os
import time
import random
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, accuracy_score, f1_score
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)
import warnings
warnings.filterwarnings("ignore")

# Configuration
TRAITS = ["cEXT", "cNEU", "cAGR", "cCON", "cOPN"]
TRAIT_NAMES = {
    "cEXT": "Extraversion",
    "cNEU": "Neuroticism",
    "cAGR": "Agreeableness",
    "cCON": "Conscientiousness",
    "cOPN": "Openness",
}

# Paths
DATA_DIR  = "../../data/split/essays"


# Choose one:
# MODEL_NAME = "microsoft/deberta-v3-base" 
MODEL_NAME = "bert-base-uncased"
# MODEL_NAME = "FacebookAI/roberta-base"

EXP_DIR   = f"../../experiment/{MODEL_NAME}"
MODEL_DIR  = f"../../model/{MODEL_NAME}"
os.makedirs(EXP_DIR,  exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

# Hyperparameters
MAX_LEN           = 512
BATCH_SIZE        = 16
EPOCHS            = 10
LEARNING_RATE     = 2e-5          # standard fine-tuning LR for BERT
WEIGHT_DECAY      = 0.01
WARMUP_RATIO      = 0.1
GRADIENT_CLIP     = 1.0
PATIENCE          = 3

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"BERT model: {MODEL_NAME}")

# Reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed(42)


In [ ]:
train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
val_df   = pd.read_csv(os.path.join(DATA_DIR, "val.csv"))
test_df  = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))

print(f"Train: {train_df.shape[0]} samples | Val: {val_df.shape[0]} samples | Test: {test_df.shape[0]} samples")
print(f"Columns: {list(train_df.columns)}")

# Class distribution per trait
print("\nClass distribution (train set):")
for trait in TRAITS:
    print(f"  {trait} ({TRAIT_NAMES[trait]}): "
          f"high={sum(train_df[trait]=='high')}, "
          f"low={sum(train_df[trait]=='low')}")

# Text length statistics
train_df["text_len"] = train_df["text"].apply(len)
print(f"\nText length stats (characters):")
print(f"  Mean: {train_df['text_len'].mean():.0f}")
print(f"  Median: {train_df['text_len'].median():.0f}")
print(f"  Max: {train_df['text_len'].max():.0f}")


In [ ]:
# Label encoding: high=1, low=0
y_train = {trait: (train_df[trait] == "high").astype(int).values for trait in TRAITS}
y_val   = {trait: (val_df[trait]   == "high").astype(int).values for trait in TRAITS}
y_test  = {trait: (test_df[trait]  == "high").astype(int).values for trait in TRAITS}

class EssayDataset(Dataset):
    """PyTorch Dataset for BERT tokenization of essays."""
    def __init__(self, texts, labels_dict, tokenizer, max_len):
        self.texts   = texts
        self.labels  = labels_dict
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        item = {
            "input_ids":      encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
        }
        if "token_type_ids" in encoding:
            item["token_type_ids"] = encoding["token_type_ids"].squeeze(0)
        for trait in TRAITS:
            item[trait] = torch.tensor(self.labels[trait][idx], dtype=torch.float)
        return item

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Create datasets
train_dataset = EssayDataset(train_df["text"].tolist(), y_train, tokenizer, MAX_LEN)
val_dataset   = EssayDataset(val_df["text"].tolist(),   y_val,   tokenizer, MAX_LEN)
test_dataset  = EssayDataset(test_df["text"].tolist(),  y_test,  tokenizer, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)} | Test batches: {len(test_loader)}")

# Inspect tokenization
sample = tokenizer(train_df["text"].iloc[0], max_length=MAX_LEN, padding="max_length", truncation=True)
print(f"\nSample tokenization:")
print(f"  Input IDs length: {len(sample['input_ids'])}")
print(f"  Num tokens: {sum(sample['attention_mask'])}")


In [ ]:
class BERTMultiLabel(nn.Module):
    def __init__(self, model_name, num_traits=5, dropout=0.1):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        hidden_size = self.bert.config.hidden_size  
        self.dropout = nn.Dropout(dropout)
        self.heads = nn.ModuleDict({
            trait: nn.Linear(hidden_size, 1) for trait in TRAITS
        })

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
        )
        # [CLS] token representation
        cls_output = outputs.last_hidden_state[:, 0, :]  # (batch, hidden)
        cls_output = self.dropout(cls_output)
        logits = {trait: self.heads[trait](cls_output).squeeze(-1) for trait in TRAITS}
        return logits

# Instantiate model
model = BERTMultiLabel(MODEL_NAME, num_traits=len(TRAITS), dropout=0.1)
model = model.to(device)
model = model.float()  # Force float32 for DeBERTa (avoids half/float dtype mismatch)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model: {MODEL_NAME} per-trait classifier")
print(f"  Total params: {total_params:,} | Trainable: {trainable_params:,}")
print(f"  Hidden size: {model.bert.config.hidden_size}")
print(f"  Num attention heads: {model.bert.config.num_attention_heads}")
print(f"  Num layers: {model.bert.config.num_hidden_layers}")

In [ ]:
def get_optimizer_grouped_params(model, lr, weight_decay):
    no_decay = ["bias", "LayerNorm.weight", "layernorm.weight"]
    optimizer_grouped_parameters = []

    # Classifier heads: highest LR
    head_params = []
    head_params_no_decay = []
    for name, param in model.named_parameters():
        if "heads" in name:
            if any(nd in name for nd in no_decay):
                head_params_no_decay.append(param)
            else:
                head_params.append(param)
    optimizer_grouped_parameters.append({"params": head_params, "lr": lr * 10, "weight_decay": weight_decay})
    optimizer_grouped_parameters.append({"params": head_params_no_decay, "lr": lr * 10, "weight_decay": 0.0})

    # BERT encoder: layer-wise LR decay
    num_layers = model.bert.config.num_hidden_layers
    for layer_idx in range(num_layers):
        layer_lr = lr * (0.95 ** (num_layers - layer_idx - 1))
        layer_params = []
        layer_params_no_decay = []
        for name, param in model.named_parameters():
            if f"bert.encoder.layer.{layer_idx}." in name:
                if any(nd in name for nd in no_decay):
                    layer_params_no_decay.append(param)
                else:
                    layer_params.append(param)
        optimizer_grouped_parameters.append({"params": layer_params, "lr": layer_lr, "weight_decay": weight_decay})
        optimizer_grouped_parameters.append({"params": layer_params_no_decay, "lr": layer_lr, "weight_decay": 0.0})

    # Embeddings: lowest LR
    embed_params = []
    embed_params_no_decay = []
    for name, param in model.named_parameters():
        if "bert.embeddings" in name:
            if any(nd in name for nd in no_decay):
                embed_params_no_decay.append(param)
            else:
                embed_params.append(param)
    optimizer_grouped_parameters.append({"params": embed_params, "lr": lr * 0.9, "weight_decay": weight_decay})
    optimizer_grouped_parameters.append({"params": embed_params_no_decay, "lr": lr * 0.9, "weight_decay": 0.0})

    return optimizer_grouped_parameters

optimizer = torch.optim.AdamW(get_optimizer_grouped_params(model, LEARNING_RATE, WEIGHT_DECAY), lr=LEARNING_RATE)

# Learning rate scheduler with warmup
total_steps     = len(train_loader) * EPOCHS
warmup_steps    = int(total_steps * WARMUP_RATIO)
scheduler       = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

criterion = nn.BCEWithLogitsLoss()

print(f"Optimizer: AdamW with layer-wise LR unfreezing")
print(f"  Base LR: {LEARNING_RATE} | Weight decay: {WEIGHT_DECAY}")
print(f"  Total steps: {total_steps} | Warmup steps: {warmup_steps}")


In [ ]:
def train_epoch(model, loader, optimizer, scheduler, criterion, device, grad_clip=1.0):
    model.train()
    total_loss = 0
    for batch in loader:
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        token_type_ids = batch.get("token_type_ids")
        if token_type_ids is not None:
            token_type_ids = token_type_ids.to(device)

        labels = {t: batch[t].to(device) for t in TRAITS}

        optimizer.zero_grad()
        logits = model(input_ids, attention_mask, token_type_ids)

        # Average BCE loss across all traits
        loss = sum(criterion(logits[t], labels[t]) for t in TRAITS) / len(TRAITS)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

    return total_loss / len(loader)


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds  = {t: [] for t in TRAITS}
    all_labels = {t: [] for t in TRAITS}

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            token_type_ids = batch.get("token_type_ids")
            if token_type_ids is not None:
                token_type_ids = token_type_ids.to(device)

            labels = {t: batch[t].to(device) for t in TRAITS}
            logits = model(input_ids, attention_mask, token_type_ids)

            loss = sum(criterion(logits[t], labels[t]) for t in TRAITS) / len(TRAITS)
            total_loss += loss.item()

            for t in TRAITS:
                probs = torch.sigmoid(logits[t]).cpu().numpy()
                all_preds[t].extend((probs >= 0.5).astype(int).tolist())
                all_labels[t].extend(labels[t].cpu().long().numpy().tolist())

    return total_loss / len(loader), all_preds, all_labels


In [ ]:
run_id = time.strftime("%Y%m%d-%H%M%S")
best_model_path = os.path.join(MODEL_DIR, f"best_model_{run_id}.pt")

history = {"epoch": [], **{t: {"train_loss": [], "val_loss": [], "val_acc": [], "val_f1": []} for t in TRAITS}}
best_val_f1 = 0.0
patience_counter = 0


print(f"\n{'='*70}")
print(f"Training {MODEL_NAME} — Run ID: {run_id}")
print(f"{'='*70}")
print(f"{'Epoch':>5} | {'Train Loss':>11} | {'Val Loss':>9} | {'Val F1 per trait':<45}")
print(f"{'-'*70}")

for epoch in range(1, EPOCHS + 1):
    train_loss = train_epoch(model, train_loader, optimizer, scheduler, criterion, device, GRADIENT_CLIP)
    val_loss, val_preds, val_labels = evaluate(model, val_loader, criterion, device)

    val_acc = {}
    val_f1  = {}
    for t in TRAITS:
        acc = np.mean(np.array(val_preds[t]) == np.array(val_labels[t]))
        f1  = f1_score(np.array(val_labels[t]), np.array(val_preds[t]), average="weighted", zero_division=0)
        val_acc[t] = acc
        val_f1[t]  = f1

    avg_val_f1 = np.mean(list(val_f1.values()))

    print(f"{epoch:>5} | {train_loss:>11.4f} | {val_loss:>9.4f} | "
          f"EXT={val_f1['cEXT']:.3f} NEU={val_f1['cNEU']:.3f} "
          f"AGR={val_f1['cAGR']:.3f} CON={val_f1['cCON']:.3f} OPN={val_f1['cOPN']:.3f} | AvgF1={avg_val_f1:.3f}")

    history["epoch"].append(epoch)
    for t in TRAITS:
        history[t]["train_loss"].append(train_loss)
        history[t]["val_loss"].append(val_loss)
        history[t]["val_acc"].append(val_acc[t])
        history[t]["val_f1"].append(val_f1[t])

    # Save best model based on average validation F1
    if avg_val_f1 > best_val_f1:
        best_val_f1 = avg_val_f1
        torch.save(model.state_dict(), best_model_path)
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"\nEarly stopping at epoch {epoch}")
            break

print(f"{'='*70}")
print(f"Best validation F1: {best_val_f1:.4f}")
print(f"Model saved to: {best_model_path}")

# Load best model for final evaluation
model.load_state_dict(torch.load(best_model_path, weights_only=True))

# Evaluate on test set
test_loss, test_preds, test_labels = evaluate(model, test_loader, criterion, device)
print(f"\nTest Loss: {test_loss:.4f}")

In [ ]:
# Compute combined classification report for all traits
gt_all   = {t: np.array(test_labels[t]) for t in TRAITS}
pred_all = {t: np.array(test_preds[t]) for t in TRAITS}

os.makedirs(EXP_DIR, exist_ok=True)

sep = "=" * 70
report_lines = [
    sep,
    "Personality Trait Detection — Classification Report",
    sep,
    f"Run ID      : {run_id}",
    f"Model       : {MODEL_NAME}",
    f"Test file   : test.csv",
    f"Test samples: {len(test_df)}",
    f"Max seq len : {MAX_LEN}",
    f"Epochs      : {len(history['epoch'])}",
    f"LR          : {LEARNING_RATE} | Batch size: {BATCH_SIZE}",
    sep,
]

for trait in TRAITS:
    gt   = gt_all[trait]
    pred = pred_all[trait]
    acc  = np.mean(gt == pred)
    f1   = f1_score(gt, pred, average="weighted", zero_division=0)

    report_str = classification_report(
        gt, pred,
        labels=[0, 1],
        target_names=["low", "high"],
        digits=4,
        zero_division=0,
    )

    report_lines.extend([
        f"\n--- {trait} ({TRAIT_NAMES[trait]}) ---",
        f"Accuracy: {acc:.4f} | F1 (weighted): {f1:.4f}",
        report_str,
    ])

report_lines.append(sep)

report_path = os.path.join(EXP_DIR, "classification_report.txt")
with open(report_path, "w", encoding="utf-8") as f:
    f.write("\n".join(report_lines))
print(f"Saved: {report_path}")

# Print per-trait summary to stdout
print(f"\n{'='*70}")
print(f"SUMMARY: Classification Results across OCEAN Traits")
print(f"{'='*70}")
print(f"{'Trait':<8} {'Test Acc':>10} {'Test F1':>10}")
print(f"{'-'*70}")

summary_data = []
for trait in TRAITS:
    gt   = gt_all[trait]
    pred = pred_all[trait]
    test_acc = np.mean(gt == pred)
    test_f1  = f1_score(gt, pred, average="weighted", zero_division=0)
    print(f"{trait:<8} {test_acc:>10.4f} {test_f1:>10.4f}")
    summary_data.append((trait, test_acc, test_f1))

print(f"{'-'*70}")
avg_test_acc = np.mean([s[1] for s in summary_data])
avg_test_f1  = np.mean([s[2] for s in summary_data])
print(f"{'AVERAGE':<8} {avg_test_acc:>10.4f} {avg_test_f1:>10.4f}")
print(f"{'='*70}")


In [ ]:
# Save raw predictions with stable filename
test_pred_df = test_df.copy()
for trait in TRAITS:
    test_pred_df[f"{trait}_pred"] = test_preds[trait]
raw_pred_path = os.path.join(EXP_DIR, "raw_predictions.csv")
test_pred_df.to_csv(raw_pred_path, index=False)
print(f"Raw predictions saved: {raw_pred_path}")

In [ ]:
# Save training config
config_path = os.path.join(MODEL_DIR, "train_config.json")
train_config = {
    "model_name": MODEL_NAME,
    "max_len": MAX_LEN,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "warmup_ratio": WARMUP_RATIO,
    "gradient_clip": GRADIENT_CLIP,
    "patience": PATIENCE,
    "epochs_trained": len(history["epoch"]),
    "best_val_f1": float(best_val_f1),
    "test_loss": float(test_loss),
    "avg_test_acc": float(avg_test_acc),
    "avg_test_f1": float(avg_test_f1),
    "run_id": run_id,
    "traits": TRAITS,
    "trait_names": TRAIT_NAMES,
}
with open(config_path, "w") as f:
    json.dump(train_config, f, indent=2)
print(f"Training config saved: {config_path}")

print(f"\nAll artifacts saved.")
print(f"  Experiment: {EXP_DIR}/")
print(f"    - classification_report.txt")
print(f"    - raw_predictions.csv")
print(f"  Model: {MODEL_DIR}/")
print(f"    - best_model_{run_id}.pt")
print(f"    - train_config.json")

In [ ]:
# # Load a saved model and run inference
# LOAD_RUN_ID = run_id  # or specify a previous run_id
# LOAD_MODEL_PATH = os.path.join(MODEL_DIR, f"model_checkpoint_{LOAD_RUN_ID}.pt")
# ckpt = torch.load(LOAD_MODEL_PATH, weights_only=False)
# print(f"Loaded config: {ckpt['config']}")

# # Re-instantiate model
# loaded_model = BERTMultiLabel(MODEL_NAME, num_traits=len(TRAITS), dropout=0.1)
# loaded_model.load_state_dict(ckpt["model_state_dict"])
# loaded_model = loaded_model.to(device)
# loaded_model.eval()

# def predict_essay(text, model, tokenizer, max_len=MAX_LEN):
#     """Predict OCEAN personality trait levels from a single essay."""
#     encoding = tokenizer(
#         text,
#         max_length=max_len,
#         padding="max_length",
#         truncation=True,
#         return_tensors="pt",
#     )
#     input_ids      = encoding["input_ids"].to(device)
#     attention_mask = encoding["attention_mask"].to(device)
#     token_type_ids = encoding.get("token_type_ids")
#     if token_type_ids is not None:
#         token_type_ids = token_type_ids.to(device)

#     with torch.no_grad():
#         logits = model(input_ids, attention_mask, token_type_ids)

#     predictions = {}
#     for trait in TRAITS:
#         prob = torch.sigmoid(logits[trait]).item()
#         predictions[trait] = "high" if prob >= 0.5 else "low"
#         predictions[f"{trait}_prob"] = round(prob, 4)
#     return predictions

# # Example predictions
# sample_essays = test_df["text"].iloc[:3].tolist()
# print("\n--- Sample predictions from loaded model ---")
# for i, essay in enumerate(sample_essays):
#     preds = predict_essay(essay, loaded_model, tokenizer)
#     print(f"\nEssay {i+1}:")
#     for t in TRAITS:
#         gt   = test_df.iloc[i][t]
#         prob = preds[f"{t}_prob"]
#         pred = preds[t]
#         match = "MATCH" if pred == gt else "MISMATCH"
#         print(f"  {t} ({TRAIT_NAMES[t]}): GT={gt}, pred={pred} (p={prob:.3f}) {match}")